# 09 – Model: Competition Prediction

### Purpose
Prediction of the number of offers using ML models.

### Steps
- Feature-Auswahl
- Modelltraining
- Evaluation
- Feature Importance


--------------------
#### Imports & Setup
-------------------

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import pickle
import json

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

# importing modules for data preparation
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# importing modules for training
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, roc_auc_score, confusion_matrix, classification_report

# importing modules for pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# importing modules for hyperparameter optimization and comparison of models
from sklearn.model_selection import validation_curve, learning_curve


In [2]:
# shut off some annoying warnings
import warnings

warnings.filterwarnings("ignore", message="A value is trying to be set on a copy")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
# ---------------------------------------------------------
# Setup style
# ---------------------------------------------------------

# show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# visualisation settings
pd.set_option('display.float_format', '{:,.2f}'.format)

In [4]:
# ---------------------------------------------------------
# Load scripts
# ----------------------------------------------------------

%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# found min directory
PROJECT_ROOT = Path("..").resolve()

# maindirectory sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# import function from script
from my_scripts.model_building import (fit_encoder, transform_with_encoder, train_model, tune_logreg)

from my_scripts.eda import (overview, filter_germany)

In [5]:
# ---------------------------------------------------------
# Load data and artifacts
# ---------------------------------------------------------

BASE_DIR = Path().resolve().parent
DATA_PATH = BASE_DIR / "data" / "dataset_nlp.pkl"
MODELS_DIR = BASE_DIR / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "risk_model.pkl"
PREPROCESSOR_PATH = MODELS_DIR / "preprocessor.pkl"
FEATURE_LIST_PATH = MODELS_DIR / "feature_list.json"
METRICS_PATH = MODELS_DIR / "metrics.json"

In [6]:
# Load dataset
with open(DATA_PATH, "rb") as f:
    df = pickle.load(f)

------------------------
# Modelling

-----------------

In [7]:
# ---------------------------------------------------------
# Create feature and test
# ---------------------------------------------------------

# delete "NUMBER_OFFERS"
df = df.drop(columns=["NUMBER_OFFERS", "TEXT_RISK_SCORE", "TEXT_RISK_LABEL"]).reset_index(drop=True)

# Load feature info
TARGET_COL = "IS_FAILED_TENDER"

features = df.drop(columns=[TARGET_COL])
target = df[TARGET_COL]

In [8]:
df.head()

,YEAR,ID_TYPE,CANCELLED,CORRECTIONS,ISO_COUNTRY_CODE,CAE_TYPE,B_AWARDED_BY_CENTRAL_BODY,TYPE_OF_CONTRACT,B_DYN_PURCH_SYST,LOTS_NUMBER,VALUE_EURO,B_EU_FUNDS,TOP_TYPE,B_ACCELERATED,OUT_OF_DIRECTIVES,CRIT_CODE,CRIT_PRICE_WEIGHT,B_ELECTRONIC_AUCTION,NUMBER_AWARDS,B_AWARDED_TO_A_GROUP,B_CONTRACTOR_SME,AWARD_VALUE_EURO,B_SUBCONTRACTED,IS_FAILED_TENDER,AWARD_MONTH,AWARD_QUARTER,DAYS_TO_AWARD,VALUE_EURO_MISSING,AWARD_VALUE_EURO_MISSING,CPV_CATEGORY,NLP_SVD_0,NLP_SVD_1,NLP_SVD_2,NLP_SVD_3,NLP_SVD_4,NLP_SVD_5,NLP_SVD_6,NLP_SVD_7,NLP_SVD_8,NLP_SVD_9,NLP_SVD_10,NLP_SVD_11,NLP_SVD_12,NLP_SVD_13,NLP_SVD_14,NLP_SVD_15,NLP_SVD_16,NLP_SVD_17,NLP_SVD_18,NLP_SVD_19,NLP_SVD_20,NLP_SVD_21,NLP_SVD_22,NLP_SVD_23,NLP_SVD_24,NLP_SVD_25,NLP_SVD_26,NLP_SVD_27,NLP_SVD_28,NLP_SVD_29,NLP_SVD_30,NLP_SVD_31,NLP_SVD_32,NLP_SVD_33,NLP_SVD_34,NLP_SVD_35,NLP_SVD_36,NLP_SVD_37,NLP_SVD_38,NLP_SVD_39,NLP_SVD_40,NLP_SVD_41,NLP_SVD_42,NLP_SVD_43,NLP_SVD_44,NLP_SVD_45,NLP_SVD_46,NLP_SVD_47,NLP_SVD_48,NLP_SVD_49,NLP_SVD_50,NLP_SVD_51,NLP_SVD_52,NLP_SVD_53,NLP_SVD_54,NLP_SVD_55,NLP_SVD_56,NLP_SVD_57,NLP_SVD_58,NLP_SVD_59,NLP_SVD_60,NLP_SVD_61,NLP_SVD_62,NLP_SVD_63,NLP_SVD_64,NLP_SVD_65,NLP_SVD_66,NLP_SVD_67,NLP_SVD_68,NLP_SVD_69,NLP_SVD_70,NLP_SVD_71,NLP_SVD_72,NLP_SVD_73,NLP_SVD_74,NLP_SVD_75,NLP_SVD_76,NLP_SVD_77,NLP_SVD_78,NLP_SVD_79,NLP_SVD_80,NLP_SVD_81,NLP_SVD_82,NLP_SVD_83,NLP_SVD_84,NLP_SVD_85,NLP_SVD_86,NLP_SVD_87,NLP_SVD_88,NLP_SVD_89,NLP_SVD_90,NLP_SVD_91,NLP_SVD_92,NLP_SVD_93,NLP_SVD_94,NLP_SVD_95,NLP_SVD_96,NLP_SVD_97,NLP_SVD_98,NLP_SVD_99,NLP_TOPIC_0,NLP_TOPIC_1,NLP_TOPIC_2,NLP_TOPIC_3,NLP_TOPIC_4,NLP_TOPIC_5,NLP_TOPIC_6,NLP_TOPIC_7,NLP_TOPIC_8,NLP_TOPIC_9,NLP_TOPIC_10,NLP_TOPIC_11,NLP_TOPIC_12,NLP_TOPIC_13,NLP_TOPIC_14
0,2008,3,0,0,DE,8,Unknown,W,0,0.00,"512,637.02",Unknown,OPE,0,0,M,100.00,0,1,0,0,"302,964.15",Unknown,0,9.00,3.00,-73.00,1,0,Other,0.23,-0.01,-0.30,0.02,-0.03,0.00,0.29,0.15,-0.06,-0.04,-0.23,-0.04,0.09,0.15,-0.01,0.05,-0.07,-0.05,-0.06,0.07,0.04,0.13,0.12,-0.20,0.10,-0.07,-0.04,-0.00,-0.05,-0.06,0.05,-0.08,-0.01,0.02,-0.02,0.03,0.05,-0.00,-0.01,-0.03,0.02,0.01,0.02,-0.03,0.00,-0.03,-0.02,0.07,0.01,-0.01,0.02,-0.02,-0.07,0.05,0.03,0.04,0.06,0.06,0.09,0.01,0.01,-0.20,0.06,-0.11,-0.01,-0.01,0.09,0.01,-0.02,-0.02,-0.04,-0.03,-0.06,0.10,0.02,-0.07,-0.03,-0.13,0.00,0.05,0.04,-0.05,0.04,-0.01,-0.01,-0.11,0.03,-0.01,-0.01,-0.04,0.03,-0.05,0.05,0.05,0.00,-0.07,0.02,0.08,-0.01,0.08,0.00,0.00,0.04,0.00,0.00,0.00,0.02,0.00,0.00,0.00,0.02,0.00,0.00,0.00,0.00
1,2008,3,0,0,DE,3,Unknown,W,0,0.00,"512,637.02",N,OPE,0,0,L,100.00,0,1,0,0,"478,780.08",N,0,12.00,4.00,-6.00,1,0,Other,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,2008,3,0,0,FR,3,Unknown,W,0,0.00,"512,637.02",N,OPE,0,0,M,100.00,0,1,0,0,"36,610.02",N,1,11.00,4.00,-37.00,1,1,Other,0.10,0.00,0.02,0.07,-0.02,-0.00,-0.00,-0.03,0.01,0.19,0.09,0.04,0.31,-0.07,-0.02,-0.03,-0.05,-0.04,-0.03,0.04,-0.03,-0.04,-0.02,0.02,0.01,-0.04,-0.05,-0.00,0.01,-0.01,-0.01,-0.02,-0.01,-0.00,-0.02,-0.00,-0.01,-0.06,0.02,0.03,0.06,0.00,0.07,0.00,0.10,0.05,0.03,0.01,0.08,-0.00,-0.07,0.02,-0.04,-0.01,-0.01,0.05,0.05,-0.00,0.02,0.00,-0.02,-0.05,-0.08,0.09,-0.01,0.01,-0.08,0.01,-0.00,-0.01,0.02,0.02,0.02,0.02,0.03,-0.03,0.01,-0.03,0.00,0.01,0.01,0.03,0.01,0.03,-0.03,0.02,-0.01,-0.04,0.02,-0.03,-0.03,0.00,0.01,0.01,0.01,0.01,0.00,-0.02,-0.02,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.04,0.00,0.00
3,2008,3,0,0,ES,1,Unknown,W,0,0.00,"512,637.02",Unknown,OPE,0,0,M,100.00,0,1,0,0,"14,237

In [9]:
features.head()

,YEAR,ID_TYPE,CANCELLED,CORRECTIONS,ISO_COUNTRY_CODE,CAE_TYPE,B_AWARDED_BY_CENTRAL_BODY,TYPE_OF_CONTRACT,B_DYN_PURCH_SYST,LOTS_NUMBER,VALUE_EURO,B_EU_FUNDS,TOP_TYPE,B_ACCELERATED,OUT_OF_DIRECTIVES,CRIT_CODE,CRIT_PRICE_WEIGHT,B_ELECTRONIC_AUCTION,NUMBER_AWARDS,B_AWARDED_TO_A_GROUP,B_CONTRACTOR_SME,AWARD_VALUE_EURO,B_SUBCONTRACTED,AWARD_MONTH,AWARD_QUARTER,DAYS_TO_AWARD,VALUE_EURO_MISSING,AWARD_VALUE_EURO_MISSING,CPV_CATEGORY,NLP_SVD_0,NLP_SVD_1,NLP_SVD_2,NLP_SVD_3,NLP_SVD_4,NLP_SVD_5,NLP_SVD_6,NLP_SVD_7,NLP_SVD_8,NLP_SVD_9,NLP_SVD_10,NLP_SVD_11,NLP_SVD_12,NLP_SVD_13,NLP_SVD_14,NLP_SVD_15,NLP_SVD_16,NLP_SVD_17,NLP_SVD_18,NLP_SVD_19,NLP_SVD_20,NLP_SVD_21,NLP_SVD_22,NLP_SVD_23,NLP_SVD_24,NLP_SVD_25,NLP_SVD_26,NLP_SVD_27,NLP_SVD_28,NLP_SVD_29,NLP_SVD_30,NLP_SVD_31,NLP_SVD_32,NLP_SVD_33,NLP_SVD_34,NLP_SVD_35,NLP_SVD_36,NLP_SVD_37,NLP_SVD_38,NLP_SVD_39,NLP_SVD_40,NLP_SVD_41,NLP_SVD_42,NLP_SVD_43,NLP_SVD_44,NLP_SVD_45,NLP_SVD_46,NLP_SVD_47,NLP_SVD_48,NLP_SVD_49,NLP_SVD_50,NLP_SVD_51,NLP_SVD_52,NLP_SVD_53,NLP_SVD_54,NLP_SVD_55,NLP_SVD_56,NLP_SVD_57,NLP_SVD_58,NLP_SVD_59,NLP_SVD_60,NLP_SVD_61,NLP_SVD_62,NLP_SVD_63,NLP_SVD_64,NLP_SVD_65,NLP_SVD_66,NLP_SVD_67,NLP_SVD_68,NLP_SVD_69,NLP_SVD_70,NLP_SVD_71,NLP_SVD_72,NLP_SVD_73,NLP_SVD_74,NLP_SVD_75,NLP_SVD_76,NLP_SVD_77,NLP_SVD_78,NLP_SVD_79,NLP_SVD_80,NLP_SVD_81,NLP_SVD_82,NLP_SVD_83,NLP_SVD_84,NLP_SVD_85,NLP_SVD_86,NLP_SVD_87,NLP_SVD_88,NLP_SVD_89,NLP_SVD_90,NLP_SVD_91,NLP_SVD_92,NLP_SVD_93,NLP_SVD_94,NLP_SVD_95,NLP_SVD_96,NLP_SVD_97,NLP_SVD_98,NLP_SVD_99,NLP_TOPIC_0,NLP_TOPIC_1,NLP_TOPIC_2,NLP_TOPIC_3,NLP_TOPIC_4,NLP_TOPIC_5,NLP_TOPIC_6,NLP_TOPIC_7,NLP_TOPIC_8,NLP_TOPIC_9,NLP_TOPIC_10,NLP_TOPIC_11,NLP_TOPIC_12,NLP_TOPIC_13,NLP_TOPIC_14
0,2008,3,0,0,DE,8,Unknown,W,0,0.00,"512,637.02",Unknown,OPE,0,0,M,100.00,0,1,0,0,"302,964.15",Unknown,9.00,3.00,-73.00,1,0,Other,0.23,-0.01,-0.30,0.02,-0.03,0.00,0.29,0.15,-0.06,-0.04,-0.23,-0.04,0.09,0.15,-0.01,0.05,-0.07,-0.05,-0.06,0.07,0.04,0.13,0.12,-0.20,0.10,-0.07,-0.04,-0.00,-0.05,-0.06,0.05,-0.08,-0.01,0.02,-0.02,0.03,0.05,-0.00,-0.01,-0.03,0.02,0.01,0.02,-0.03,0.00,-0.03,-0.02,0.07,0.01,-0.01,0.02,-0.02,-0.07,0.05,0.03,0.04,0.06,0.06,0.09,0.01,0.01,-0.20,0.06,-0.11,-0.01,-0.01,0.09,0.01,-0.02,-0.02,-0.04,-0.03,-0.06,0.10,0.02,-0.07,-0.03,-0.13,0.00,0.05,0.04,-0.05,0.04,-0.01,-0.01,-0.11,0.03,-0.01,-0.01,-0.04,0.03,-0.05,0.05,0.05,0.00,-0.07,0.02,0.08,-0.01,0.08,0.00,0.00,0.04,0.00,0.00,0.00,0.02,0.00,0.00,0.00,0.02,0.00,0.00,0.00,0.00
1,2008,3,0,0,DE,3,Unknown,W,0,0.00,"512,637.02",N,OPE,0,0,L,100.00,0,1,0,0,"478,780.08",N,12.00,4.00,-6.00,1,0,Other,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,2008,3,0,0,FR,3,Unknown,W,0,0.00,"512,637.02",N,OPE,0,0,M,100.00,0,1,0,0,"36,610.02",N,11.00,4.00,-37.00,1,1,Other,0.10,0.00,0.02,0.07,-0.02,-0.00,-0.00,-0.03,0.01,0.19,0.09,0.04,0.31,-0.07,-0.02,-0.03,-0.05,-0.04,-0.03,0.04,-0.03,-0.04,-0.02,0.02,0.01,-0.04,-0.05,-0.00,0.01,-0.01,-0.01,-0.02,-0.01,-0.00,-0.02,-0.00,-0.01,-0.06,0.02,0.03,0.06,0.00,0.07,0.00,0.10,0.05,0.03,0.01,0.08,-0.00,-0.07,0.02,-0.04,-0.01,-0.01,0.05,0.05,-0.00,0.02,0.00,-0.02,-0.05,-0.08,0.09,-0.01,0.01,-0.08,0.01,-0.00,-0.01,0.02,0.02,0.02,0.02,0.03,-0.03,0.01,-0.03,0.00,0.01,0.01,0.03,0.01,0.03,-0.03,0.02,-0.01,-0.04,0.02,-0.03,-0.03,0.00,0.01,0.01,0.01,0.01,0.00,-0.02,-0.02,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.04,0.00,0.00
3,2008,3,0,0,ES,1,Unknown,W,0,0.00,"512,637.02",Unknown,OPE,0,0,M,100.00,0,1,0,0,"14,237,368.10",Unknown,10.00,

In [10]:
# ---------------------------------------------------------
# Categorical features encoding
# ---------------------------------------------------------

# fit encoder only on train
encoder = fit_encoder(features)

# transform train and test
features = transform_with_encoder(features, encoder)


In [11]:
# Train/test split

features_train, features_test, target_train, target_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42,
    stratify=target)

-------------------------------------------------------------------
### Model pipeline

-------------------------------------------------------------------

In [12]:
# Logistic Regression
lr_model = train_model(LogisticRegression(max_iter=500, class_weight='balanced'), features_train, features_test, target_train, target_test)


=== Model: LogisticRegression ===
F1: 0.5320359171030312
ROC-AUC: 0.764397219861664
              precision    recall  f1-score   support

           0       0.88      0.72      0.79    534221
           1       0.44      0.68      0.53    170104

    accuracy                           0.71    704325
   macro avg       0.66      0.70      0.66    704325
weighted avg       0.77      0.71      0.73    704325



In [14]:
# Random Forest
rf_model = train_model(RandomForestClassifier(n_estimators=50, max_depth=10, min_samples_split=4, random_state=42, class_weight='balanced'), 
                       features_train, features_test, target_train, target_test)


=== Model: RandomForestClassifier ===
F1: 0.5405891526266946
ROC-AUC: 0.7729554634007076
              precision    recall  f1-score   support

           0       0.87      0.75      0.81    534221
           1       0.46      0.66      0.54    170104

    accuracy                           0.73    704325
   macro avg       0.67      0.71      0.67    704325
weighted avg       0.77      0.73      0.74    704325



In [ ]:
# Gradient Boosting
gb_model = train_model(GradientBoostingClassifier(n_estimators=50, learning_rate=0.05, max_depth=3, random_state=42), 
                       features_train, features_test, target_train, target_test)

In [ ]:
# Hyperparameter tuning
best_lr = tune_logreg(features_train, target_train)

In [ ]:
cat_model = train_model(CatBoostClassifier(iterations=50,learning_rate=0.05,depth=6, loss_function="Logloss", verbose=False, random_state=42),
                        features_train, features_test, target_train, target_test)

# Model evaluation

In [ ]:
def evaluate(pred, proba, name):
    print(f"\n=== {name} ===")
    print("Accuracy:", accuracy_score(target_test, pred))
    print("F1:", f1_score(target_test, pred))
    print("ROC-AUC:", roc_auc_score(target_test, proba))
    print("Confusion matrix:\n", confusion_matrix(target_test, pred))

evaluate(pred_lr, proba_lr, "Logistic Regression")
evaluate(pred_rf, proba_rf, "Random Forest")
evaluate(pred_gb, proba_gb, "Gradient Boosting")


In [ ]:
# ROC Curve

plot_roc_curve(
    model=clf,
    features_test=features_test,
    target_test=target_test,
    save=True
)

In [ ]:
# Precision–Recall Curve

plot_pr_curve(
    model=clf,
    features_test=features_test,
    target_test=target_test,
    save=True
)


In [ ]:
# Confusion Matrix

plot_confusion_matrix(
    model=clf,
    features_test=features_test,
    target_test=target_test,
    save=True
)


In [ ]:
# Plot

plot_feature_importance(
    model=clf,
    feature_names=feature_names,
    save=True
)


In [ ]:
# Save model
joblib.dump(clf, MODEL_PATH)

# Save preprocessor separately (optional)
joblib.dump(preprocessor, PREPROCESSOR_PATH)

# Save feature list
feature_list = {
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "all_feature_cols": features.columns.tolist(),
    "target_col": TARGET_COL,
}

with open(FEATURE_LIST_PATH, "w", encoding="utf-8") as f:
    json.dump(feature_list, f, indent=2)

# Save metrics
with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
